In [20]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import numpy as np
from sklearn.preprocessing import LabelEncoder

In [35]:
# Load data
df = pd.read_csv('/Users/gautam/Desktop/MinneMUDAC2025/updated_mathclength_sorted_Training.csv', low_memory=False)
df = df.dropna(subset=['Completion Date', 'Match Support Contact Notes'])
df['Completion Date'] = pd.to_datetime(df['Completion Date'])
static_columns = [
    'Big Age', 
    'Big Gender', 
    'Big Race/Ethnicity',
    'Little Gender', 
    'Little Participant: Race/Ethnicity',
    'Program', 
    'Program Type',
]

In [36]:
# Prepare Label Encoders
label_encoders = {}
for col in static_columns:
    if df[col].dtype == 'object':
        le = LabelEncoder()
        df[col] = df[col].fillna('Unknown')  # Handle NaNs
        df[col] = le.fit_transform(df[col])
        label_encoders[col] = le
    else:
        df[col] = df[col].fillna(df[col].mean())  # Fill numeric NaN

In [37]:
# Group and sort by Match ID and Completion Date
grouped = df.groupby('Match ID 18Char')

In [38]:
# Prepare data tuples: (sequence of notes, numerical features, target final match length)
data = []
for match_id, group in grouped:
    group_sorted = group.sort_values(by='Completion Date')
    notes_sequence = group_sorted['Match Support Contact Notes'].tolist()

    # Your time-dependent features (aggregated)
    avg_match_length = group_sorted['Match Length'].mean()
    num_contacts = len(group_sorted)

    # STATIC features: pull once from the first row (or customize)
    static_values = group_sorted.iloc[0][static_columns].values.astype(float)
    
    # Combine numerical and static features
    combined_features = [avg_match_length, num_contacts] + static_values.tolist()
    final_match_length = group_sorted['Match Length'].iloc[-1]
    
    data.append((notes_sequence, combined_features, final_match_length))


In [39]:

# Initialize SentenceTransformer model for text embedding
sbert = SentenceTransformer('all-MiniLM-L6-v2')

In [40]:
# Custom Dataset
class MatchDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        notes, features, target = self.data[idx]
        embeddings = sbert.encode(notes)  # Shape: (seq_len, embed_dim)
        features = torch.tensor(features, dtype=torch.float32)
        return torch.tensor(embeddings, dtype=torch.float32), features, torch.tensor(target, dtype=torch.float32)


In [28]:


# Collate function to handle variable sequence lengths
def collate_fn(batch):
    sequences, features, targets = zip(*batch)
    lengths = [seq.shape[0] for seq in sequences]
    padded_sequences = nn.utils.rnn.pad_sequence(sequences, batch_first=True)
    features = torch.stack(features)
    return padded_sequences, torch.tensor(lengths), features, torch.tensor(targets)


In [29]:

# Model definition
class SentimentRNN(nn.Module):
    def __init__(self, embed_dim, hidden_dim, feature_dim):
        super(SentimentRNN, self).__init__()
        self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim + feature_dim, 1)

    def forward(self, x, lengths, features):
        packed_input = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_output, hidden = self.rnn(packed_input)
        combined = torch.cat([hidden[-1], features], dim=1)
        output = self.fc(combined)
        return output.squeeze()


In [41]:
# Hyperparameters
embed_dim = 384  # Embedding size of 'all-MiniLM-L6-v2'
hidden_dim = 128
feature_dim = 2+ len(static_columns)  # Number of additional numeric features
batch_size = 8

In [42]:

# Dataset and DataLoader
dataset = MatchDataset(data)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)


In [44]:
# Model, Loss, Optimizer
model = SentimentRNN(embed_dim, hidden_dim, feature_dim)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [45]:


# Training loop
num_epochs = 5
model.train()
for epoch in range(num_epochs):
    epoch_loss = 0
    preds_list = []
    targets_list = []
    for padded_seqs, lengths, features, targets in tqdm(train_loader):
        optimizer.zero_grad()
        preds = model(padded_seqs, lengths, features)
        loss = criterion(preds, targets)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        preds_list.append(preds.detach().numpy())
        targets_list.append(targets.numpy())

    # Calculate RMSE at the end of the epoch
    preds_all = np.concatenate(preds_list)
    targets_all = np.concatenate(targets_list)
    rmse = np.sqrt(np.mean((preds_all - targets_all) ** 2))

    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {epoch_loss / len(train_loader):.4f}, Train RMSE: {rmse:.4f}")

100%|██████████| 327/327 [03:08<00:00,  1.74it/s]


Epoch 1/5, Train Loss: 179.1373, Train RMSE: 13.3934


100%|██████████| 327/327 [03:57<00:00,  1.38it/s]


Epoch 2/5, Train Loss: 54.6000, Train RMSE: 7.3936


100%|██████████| 327/327 [04:04<00:00,  1.34it/s]


Epoch 3/5, Train Loss: 37.1206, Train RMSE: 6.0949


100%|██████████| 327/327 [04:12<00:00,  1.30it/s]


Epoch 4/5, Train Loss: 29.9580, Train RMSE: 5.4782


100%|██████████| 327/327 [04:22<00:00,  1.25it/s]

Epoch 5/5, Train Loss: 24.9213, Train RMSE: 4.9898


In [46]:

# Evaluation on test set
model.eval()
preds_list = []
targets_list = []
with torch.no_grad():
    for padded_seqs, lengths, features, targets in test_loader:
        preds = model(padded_seqs, lengths, features)
        preds_list.append(preds.numpy())
        targets_list.append(targets.numpy())

preds_all = np.concatenate(preds_list)
targets_all = np.concatenate(targets_list)
test_rmse = np.sqrt(np.mean((preds_all - targets_all) ** 2))
print(f"Test RMSE: {test_rmse:.4f}")

Test RMSE: 4.8315
